# Notebook 4: Custom Tools & Agent Reasoning

**🎯 Goal:** Learn how to create your own custom tools, handle complex inputs, and understand the agent's internal reasoning process (the ReAct pattern).

## 🧩 Concept Overview

Agents are only as powerful as the tools they can access. While LangChain has many built-in tools, the real power comes from creating your own.

- **Custom Tool:** Any Python function can be turned into a tool. This allows you to connect an agent to any API, database, or custom logic you can imagine.
- **The Description is Everything:** The most important part of a tool is its **docstring description**. The LLM does not see your tool's code; it only sees its name, arguments, and description. A good description is crucial for the agent to know when and how to use the tool.
- **ReAct Pattern:** This stands for **Reason + Act**. It's the framework that agents use to decide what to do next. By setting `verbose=True`, we can watch this process unfold.

## 🖼️ Visual Diagram: Why Descriptions Matter

```
                                     +-----------------------------------------+
LLM's Brain: "User wants to know    | Tool Name: 'web_search'                 |
the capital of France. Which tool   | Args: 'query' (string)                  |
should I use?"                     | Description: 'Searches the web for up-  |
                                     | to-date information.'                   |
                                     +-----------------------------------------+
                                                        ^
                                                        | This looks like the right tool!
                                                        |
                                     +-----------------------------------------+
                                     | Tool Name: 'calculator'                 |
                                     | Args: 'expression' (string)             |
                                     | Description: 'Evaluates math problems.' |
                                     +-----------------------------------------+
```

## 1. Creating Custom Tools

The easiest way to create a tool is with the `@tool` decorator. It automatically infers the tool's name and arguments from your Python function.

In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> float:
    """Evaluate a math expression. The input should be a valid mathematical expression string."""
    print(f"-- Calling Calculator with: {expression} --")
    return eval(expression)

@tool
def search_web(query: str) -> str:
    """Search the web for a query. Use this for finding current events or up-to-date information."""
    print(f"-- Calling Web Search with: {query} --")
    return f"Web results for '{query}': [mock result] LangChain was created in 2022."

# Let's inspect the tools
print(f"Calculator Tool: {calculator.name}, Description: {calculator.description}")
print(f"Web Search Tool: {search_web.name}, Description: {search_web.description}")

### Advanced Tools: Structured Inputs & Error Handling

Sometimes, you need more control. You can use a Pydantic `BaseModel` to define structured arguments for your tool. This is more robust and provides better validation.

In [ ]:
from pydantic import BaseModel, Field

# Define the input schema for the tool
class SafeCalculatorInput(BaseModel):
    expression: str = Field(description="A mathematical expression like '2+2' or '5*4'.")

# Use the `args_schema` to apply the Pydantic model
@tool(args_schema=SafeCalculatorInput)
def safe_calculator(expression: str) -> float:
    """A safe calculator that evaluates a math expression. Use this for any math questions."""
    try:
        return eval(expression)
    except Exception as e:
        # It's good practice for tools to handle their own errors
        return f"Error evaluating expression: {e}"

print(f"Safe Calculator Args: {safe_calculator.args}")

## 2. The Agent's Thought Process (ReAct)

Let's build an agent with our new tools and watch its reasoning process. The `verbose=True` flag is our window into the agent's mind.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain import hub

llm = ChatOpenAI(model="gpt-4o", temperature=0)
tools = [safe_calculator, search_web]

# The ReAct prompt has placeholders for tools and the agent's scratchpad (its thoughts)
prompt = hub.pull("hwchase17/react")

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({"input": "When was LangChain created, and what is that year multiplied by 10?"})

### Breaking Down the ReAct Log

When you run the cell above, you'll see output like this:

```
Thought: The user is asking two questions. First, when was LangChain created. Second, what is that year multiplied by 10. I need to use the web search tool to find the creation year of LangChain first.
Action: search_web
Action Input: "When was LangChain created?"
Observation: Web results for 'When was LangChain created?': [mock result] LangChain was created in 2022.
Thought: I have found that LangChain was created in 2022. Now I need to multiply this year by 10. I should use the calculator tool for this.
Action: safe_calculator
Action Input: expression='2022*10'
Observation: 20220
Thought: I have the final answer.
Final Answer: LangChain was created in 2022, and that year multiplied by 10 is 20220.
```
This loop of `Thought -> Action -> Observation` is the essence of how ReAct agents work.

## 🔧 Hands-On Exercise

**Goal:** Create a custom tool with a Pydantic schema and use it in an agent.

1.  Create a Pydantic `BaseModel` called `GreetingInput` with two string fields: `name` and `city`.
2.  Create a tool named `create_greeting` that uses the `GreetingInput` schema.
3.  The tool should return a greeting string, e.g., `"Hello, Alice from London!"`
4.  Build an agent with this tool and ask it: `"Greet Jules who lives in New York."`

In [ ]:
# 1. Create the Pydantic model
class GreetingInput(BaseModel):
    name: str = Field(description="The name of the person to greet.")
    city: str = Field(description="The city where the person lives.")

# 2. Create the tool
@tool(args_schema=GreetingInput)
def create_greeting(name: str, city: str) -> str:
    """Creates a personalized greeting for a person based on their name and city."""
    return f"Hello, {name} from {city}!"

# 3. Build the agent
greeting_tools = [create_greeting]
greeting_agent = create_react_agent(llm, greeting_tools, prompt)
greeting_executor = AgentExecutor(agent=greeting_agent, tools=greeting_tools, verbose=True)

# 4. Run it!
greeting_executor.invoke({"input": "Greet Jules who lives in New York."})

## 🐞 Bug Bounty

The agent below has a perfectly functional tool, but the **docstring description is terrible**. Because the description is unhelpful, the agent has no idea when to use the tool.

**Task:** Fix the `docstring` of the `buggy_weather_tool` so the agent can correctly answer the question.

In [ ]:
# --- BROKEN CODE ---
@tool
def buggy_weather_tool(query: str) -> str:
    """Takes a string and returns a string.""" # <-- Bad description!
    return f"The weather in {query} is 72 and sunny."

buggy_tools = [buggy_weather_tool]
buggy_agent = create_react_agent(llm, buggy_tools, prompt)
buggy_executor = AgentExecutor(agent=buggy_agent, tools=buggy_tools, verbose=True)

print("--- Running with buggy tool ---")
# The agent will likely fail here because it doesn't know the tool is for weather.
buggy_executor.invoke({"input": "What's the weather in San Francisco?"})


# --- FIXED CODE ---
@tool
def fixed_weather_tool(city: str) -> str:
    """Use this tool to get the current weather for a specific city."""
    return f"The weather in {city} is 72 and sunny."

fixed_tools = [fixed_weather_tool]
fixed_agent = create_react_agent(llm, fixed_tools, prompt)
fixed_executor = AgentExecutor(agent=fixed_agent, tools=fixed_tools, verbose=True)

print("\n--- Running with fixed tool ---")
fixed_executor.invoke({"input": "What's the weather in San Francisco?"})

## 💡 Pro Tip

**Spend more time writing good tool descriptions than writing the tool's code.** The LLM never sees your Python code. Its entire decision-making process is based on the tool's name, its arguments, and its description. A well-described tool is the most important factor in building a reliable agent.

## 🏁 Summary

You can now build powerful, custom agents that can reason and act.

1.  **Any Function Can Be a Tool:** The `@tool` decorator makes it easy to give agents new capabilities.
2.  **Descriptions are Mission-Critical:** The agent's ability to solve a problem correctly depends almost entirely on how well you describe the tools it can use.
3.  **The ReAct Loop is Transparent:** Using `verbose=True` lets you see the agent's step-by-step reasoning, making it easy to debug and understand its behavior.